# 06_predict_submit — LGBMRankerで全顧客分の提出ファイルを作成(optunaチューニング版)

`07_optuna_tune.ipynb` で学習したLGBMRanker(5候補ソース×学習4週+optuna20試行でのハイパラ探索、
バックテストMAP@12=0.03357、前回提出の0.0332/Public LB 0.02556から微増)を使い、
cutoff=学習データ最終日(全データ)で実際の提出物を作る。

- 直近56日以内にアクティブな顧客(約40万人): 候補プール→特徴量→Rankerでスコアリングし上位12件
- それ以外(コールドスタート/長期非アクティブ、約100万人): モデルの手がかりが無いため、直近1週間のトレンド人気トップ12でフォールバック(02_baselineと同じ考え方)

In [ ]:
import os
import sys
import glob
import pickle

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

sys.path.append(os.path.abspath("."))
from common import DATA_DIR, SUBMISSION_DIR, active_customers, format_prediction, get_con
from candidates import build_popularity_candidates
from features import build_feature_table

MODELS_DIR = os.path.join("..", "02_output", "models")
FEATURES_DIR = os.path.join("..", "02_output", "features")

# 現在のベスト(Public LB=0.02624, experiment_log.csvのrun_20260901_0953_ranker、6ソース・デフォルトパラメータ)。
# 単純に最新ファイルを使うと、性能が悪化した実験(例: optunaのval週過適合、from_price_band追加も同様に悪化)を
# 拾ってしまうため固定する。更新する場合は、新しいrunがexperiment_log.csvでこれを上回ったことを確認してから変更すること。
BACKTEST_MAP12 = 0.0338
ranker_path = os.path.join(MODELS_DIR, "model_20260901_0953_ranker.pkl")
print(f"model: {ranker_path}")
with open(ranker_path, "rb") as f:
    payload = pickle.load(f)
ranker = payload["model"]
feature_cols = payload["feature_cols"]
cat_cols = payload["cat_cols"]
nullable_int_cols = ["u_days_since_last_purchase", "ui_days_since_last_bought",
                      "ui_days_since_last_bought_product_code"]

con = get_con()
final_cutoff = con.execute("SELECT MAX(t_dat) FROM trans").fetchone()[0].strftime("%Y-%m-%d")
print(f"final_cutoff = {final_cutoff}")

## カテゴリ列のcategories復元

学習時(05)にtrain_features側の`category`dtypeで学習しているため、予測時も同じcategories(コード対応)を使う必要がある。
04で保存したtrain_features parquetから、05と同じくpyarrowのdictionary encoding経由でcategoriesを再現する
(pandasの素朴な読み込みだとカテゴリ列が巨大なobject dtypeとして展開されメモリを圧迫するため)。

In [ ]:
train_path = sorted(glob.glob(os.path.join(FEATURES_DIR, "train_features_*.parquet")))[-1]
print(f"train (categories復元用): {train_path}")
train_cat_ref = pq.read_table(train_path, columns=cat_cols).to_pandas(categories=cat_cols)
train_categories = {c: train_cat_ref[c].cat.categories for c in cat_cols}
del train_cat_ref

## 対象顧客の絞り込みと候補+特徴量の構築

直近56日アクティブな顧客のみを対象にRankerでスコアリングする(04と同じ絞り込み方針)。

In [ ]:
target_customers = active_customers(con, final_cutoff, trailing_days=56)
print(f"アクティブ顧客数(Ranker予測対象): {len(target_customers):,}")

final_df = build_feature_table(con, final_cutoff, trailing_days=56, with_label=False)
print(f"final_df shape: {final_df.shape}")

In [ ]:
for c in nullable_int_cols:
    final_df[c] = final_df[c].astype("float32")
for c in cat_cols:
    final_df[c] = pd.Categorical(final_df[c], categories=train_categories[c])

X_final = final_df[feature_cols]
final_df["score"] = ranker.predict(X_final)

## アクティブ顧客: Rankerの上位12件

In [ ]:
top12 = (
    final_df[["customer_idx", "article_id", "score"]]
    .sort_values("score", ascending=False)
    .groupby("customer_idx")
    .head(12)
)
pred_map_active = top12.groupby("customer_idx")["article_id"].apply(list).to_dict()

n_full = sum(1 for v in pred_map_active.values() if len(v) == 12)
print(f"Ranker予測を持つ顧客数: {len(pred_map_active):,}(うち12件ちょうど: {n_full:,})")

## コールドスタート/非アクティブ顧客: トレンド人気フォールバック

In [ ]:
pop_list_final = build_popularity_candidates(con, final_cutoff, top_n=12)
print(f"トレンド人気トップ12: {pop_list_final}")

all_customers = con.execute("SELECT customer_id, customer_idx FROM cust_map").df()
print(f"全顧客数: {len(all_customers):,}")


def get_prediction(customer_idx):
    candidates = pred_map_active.get(customer_idx)
    if candidates and len(candidates) >= 12:
        return candidates[:12]
    if candidates:
        seen = set(candidates)
        filled = list(candidates)
        for a in pop_list_final:
            if a not in seen:
                filled.append(a)
                seen.add(a)
            if len(filled) >= 12:
                break
        return filled[:12]
    return pop_list_final


all_customers["prediction"] = all_customers["customer_idx"].apply(get_prediction).apply(format_prediction)
submission = all_customers[["customer_id", "prediction"]]
submission.head()

## フォーマット検証

In [ ]:
assert len(submission) == len(all_customers)
assert submission["prediction"].apply(lambda s: len(s.split())).eq(12).all(), "12件に満たない顧客がいます"

sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"), usecols=["customer_id"])
assert set(submission["customer_id"]) == set(sample_submission["customer_id"]), "sample_submission.csvの顧客と一致しません"
print("フォーマット検証OK")

## 保存

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
submission_filename = f"submission_{timestamp}_cv{BACKTEST_MAP12:.4f}.csv"
submission_path = os.path.join(SUBMISSION_DIR, submission_filename)
submission.to_csv(submission_path, index=False)
print(f"保存しました: {submission_path}")
print(submission_filename)